# OrganAMNIST Model Training Part-2

## Setup and Reproducibility

!pip install -q "numpy==1.26.4" "scipy==1.13.1"

!pip install -q "transformers==4.44.2" --no-deps

!pip install -q "tokenizers>=0.19,<0.20" "huggingface-hub>=0.23,<0.25" "safetensors>=0.4"

!pip install -q "medmnist" "tf-keras"

In [1]:
# SETUP AND REPRODUCIBILITY

import os
os.environ["TF_USE_LEGACY_KERAS"] = "1"
os.environ['PYTHONHASHSEED'] = '42'

import random
import numpy as np
import tensorflow as tf

# Global reproducibility seed
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)
tf.keras.utils.set_random_seed(SEED)
tf.config.experimental.enable_op_determinism()

# Imports
import medmnist
from medmnist import INFO
import gc
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from tensorflow.keras import layers, models, applications, mixed_precision, callbacks
from tensorflow.keras.utils import to_categorical
from tensorflow.keras.callbacks import ModelCheckpoint, EarlyStopping, ReduceLROnPlateau
from transformers import TFViTForImageClassification
from sklearn.metrics import (classification_report, confusion_matrix, accuracy_score,
                              roc_auc_score, precision_score, recall_score, f1_score,
                              average_precision_score, log_loss)
from sklearn.calibration import calibration_curve
from scipy.signal import find_peaks
from scipy.stats import gaussian_kde
import warnings
warnings.filterwarnings('ignore')

print(f"NumPy: {np.__version__}")
print(f"TensorFlow: {tf.__version__}")
print(f"Random Seed: {SEED}")
print(f"Legacy Keras: {os.environ.get('TF_USE_LEGACY_KERAS')}")

2026-06-12 04:41:25.888586: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1781239286.134646     132 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1781239286.205401     132 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1781239286.774939     132 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1781239286.774985     132 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1781239286.774988     132 computation_placer.cc:177] computation placer alr

NumPy: 1.26.4
TensorFlow: 2.19.0
Random Seed: 42
Legacy Keras: 1


## Configuration

In [2]:
# CONFIGURATION

DATASET = 'organamnist'
IMAGE_SIZE = 224
BATCH_SIZE = 8 #incresed batch size for OrganAMNIST because of the lazy loading of the data
EPOCHS = 30
LR_VIT = 2e-5
LR_CNN = 1e-4
DROPOUT_META = 0.5
META_BATCH_SIZE = 16
META_EPOCHS = 50
META_PATIENCE = 5
PATIENCE_EARLY = 7
PATIENCE_REDUCE = 3
REDUCE_FACTOR = 0.5
MIN_LR = 1e-6

OUTPUT_DIR = f"/kaggle/working/{DATASET}_outputs"
os.makedirs(OUTPUT_DIR, exist_ok=True)

# Mixed precision for speed
policy = mixed_precision.Policy('mixed_float16')
mixed_precision.set_global_policy(policy)
print(f"Mixed Precision: {policy.name}")
print(f"Output Dir: {OUTPUT_DIR}")

INFO:tensorflow:Mixed precision compatibility check (mixed_float16): OK
Your GPUs will likely run quickly with dtype policy mixed_float16 as they all have compute capability of at least 7.0
Mixed Precision: mixed_float16
Output Dir: /kaggle/working/organamnist_outputs


## Data Loading

In [3]:
# DATA LOADER
# FOR ORGANAMNIST ONLY: LAZY LOADING VERSION


NATIVE_LOAD_SIZE = 128 

def load_medmnist_data(dataset_flag, target_size=224, batch_size=8,
                       native_size=NATIVE_LOAD_SIZE):
    """
    Memory-efficient lazy-loading data loader.
    
    Loads all splits at native_size uint8 from MedMNIST+
    Resizes to target_size and casts to float32 in the tf.data pipeline.
    
    Defaults:
      native_size=128 → ~4 GB memory, near-native quality
      Use native_size=64 if memory still tight (~1 GB total)
      Use native_size=224 if you have >16 GB free RAM (no resize)
    """
    info = INFO[dataset_flag]
    n_classes = len(info['label'])
    DataClass = getattr(medmnist, info['python_class'])

    print(f"Loading {dataset_flag} at native {native_size}x{native_size}...")
    print(f"  Lazy resize to {target_size}x{target_size} in pipeline")
    
    train_data = DataClass(split='train', download=True, size=native_size)
    val_data = DataClass(split='val', download=True, size=native_size)
    test_data = DataClass(split='test', download=True, size=native_size)

    X_train, y_train = train_data.imgs, train_data.labels
    X_val, y_val = val_data.imgs, val_data.labels
    X_test, y_test = test_data.imgs, test_data.labels

    # Grayscale to RGB
    if X_train.shape[-1] != 3 and len(X_train.shape) == 3:
        X_train = np.repeat(X_train[..., np.newaxis], 3, -1)
        X_val = np.repeat(X_val[..., np.newaxis], 3, -1)
        X_test = np.repeat(X_test[..., np.newaxis], 3, -1)

    print(f"  Memory footprint:")
    print(f"    Train: {X_train.nbytes / 1024**2:.1f} MB ({X_train.shape})")
    print(f"    Val:   {X_val.nbytes / 1024**2:.1f} MB ({X_val.shape})")
    print(f"    Test:  {X_test.nbytes / 1024**2:.1f} MB ({X_test.shape})")

    y_train_enc = to_categorical(y_train, n_classes)
    y_val_enc = to_categorical(y_val, n_classes)
    y_test_enc = to_categorical(y_test, n_classes)

    def preprocess(image, label):
        image = tf.cast(image, tf.float32)
        # Only resize if needed (skip if native_size == target_size)
        if native_size != target_size:
            image = tf.image.resize(image, [target_size, target_size], method='bilinear')
        return image, label

    train_ds = (tf.data.Dataset.from_tensor_slices((X_train, y_train_enc))
                .shuffle(1000, seed=SEED)
                .map(preprocess, num_parallel_calls=tf.data.AUTOTUNE)
                .batch(batch_size)
                .prefetch(tf.data.AUTOTUNE))
    
    val_ds = (tf.data.Dataset.from_tensor_slices((X_val, y_val_enc))
              .map(preprocess, num_parallel_calls=tf.data.AUTOTUNE)
              .batch(batch_size)
              .prefetch(tf.data.AUTOTUNE))
    
    test_ds = (tf.data.Dataset.from_tensor_slices((X_test, y_test_enc))
               .map(preprocess, num_parallel_calls=tf.data.AUTOTUNE)
               .batch(batch_size)
               .prefetch(tf.data.AUTOTUNE))

    return train_ds, val_ds, test_ds, n_classes, y_test, X_test, info


# Load with lazy pipeline
train_ds, val_ds, test_ds, n_classes, y_test_true, X_test_raw, dataset_info = load_medmnist_data(DATASET, target_size=IMAGE_SIZE, batch_size=BATCH_SIZE)
y_true_flat = y_test_true.flatten()
y_true_cat = to_categorical(y_true_flat, n_classes)
CLASS_NAMES = [dataset_info['label'][str(i)] for i in range(n_classes)]
print(f"\nClasses: {n_classes}")
print(f"Class Names: {CLASS_NAMES}")
print(f"Test set size: {len(y_true_flat)}")

Loading organamnist at native 128x128...
  Lazy resize to 224x224 in pipeline


100%|██████████| 708M/708M [00:43<00:00, 16.4MB/s] 


  Memory footprint:
    Train: 1620.0 MB ((34561, 128, 128, 3))
    Val:   304.3 MB ((6491, 128, 128, 3))
    Test:  833.3 MB ((17778, 128, 128, 3))


I0000 00:00:1781239374.687848     132 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 13757 MB memory:  -> device: 0, name: Tesla T4, pci bus id: 0000:00:04.0, compute capability: 7.5
I0000 00:00:1781239374.693986     132 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:1 with 13757 MB memory:  -> device: 1, name: Tesla T4, pci bus id: 0000:00:05.0, compute capability: 7.5



Classes: 11
Class Names: ['bladder', 'femur-left', 'femur-right', 'heart', 'kidney-left', 'kidney-right', 'liver', 'lung-left', 'lung-right', 'pancreas', 'spleen']
Test set size: 17778


## Class Distribuition Analysis

In [5]:
# CLASS DISTRIBUTION ANALYSIS

DataClass = getattr(medmnist, dataset_info['python_class'])
train_labels = DataClass(split='train', size=NATIVE_LOAD_SIZE).labels.flatten()
val_labels = DataClass(split='val', size=NATIVE_LOAD_SIZE).labels.flatten()
test_labels = DataClass(split='test', size=NATIVE_LOAD_SIZE).labels.flatten()

dist_rows = []
for i, name in enumerate(CLASS_NAMES):
    train_count = (train_labels == i).sum()
    val_count = (val_labels == i).sum()
    test_count = (test_labels == i).sum()
    total = train_count + val_count + test_count
    dist_rows.append({
        'Class ID': i,
        'Class Name': name,
        'Train': train_count,
        'Val': val_count,
        'Test': test_count,
        'Total': total,
        '% of Total': round(100 * total / (len(train_labels)+len(val_labels)+len(test_labels)), 2)
    })

dist_df = pd.DataFrame(dist_rows)
print("\n=== CLASS DISTRIBUTION TABLE ===")
print(dist_df.to_string(index=False))

ir = dist_df['Train'].max() / dist_df['Train'].min()
print(f"\nImbalance Ratio (IR): {ir:.2f}")
print(f"Total samples: Train={len(train_labels)}, Val={len(val_labels)}, Test={len(test_labels)}")
print(f"Missing values check: Train NaNs={pd.isna(train_labels).sum()}, Test NaNs={pd.isna(test_labels).sum()}")
print(f"Duplicates: handled by MedMNIST v2 official splits (no overlap between train/val/test)")

dist_df.to_csv(f"{OUTPUT_DIR}/class_distribution.csv", index=False)


=== CLASS DISTRIBUTION TABLE ===
 Class ID   Class Name  Train  Val  Test  Total  % of Total
        0      bladder   1956  321  1036   3313        5.63
        1   femur-left   1390  233   784   2407        4.09
        2  femur-right   1357  225   793   2375        4.04
        3        heart   1474  392   785   2651        4.51
        4  kidney-left   3963  568  2064   6595       11.21
        5 kidney-right   3817  637  1965   6419       10.91
        6        liver   6164 1033  3285  10482       17.82
        7    lung-left   3919 1033  1747   6699       11.39
        8   lung-right   3929 1009  1813   6751       11.48
        9     pancreas   3031  529  1622   5182        8.81
       10       spleen   3561  511  1884   5956       10.12

Imbalance Ratio (IR): 4.54
Total samples: Train=34561, Val=6491, Test=17778
Missing values check: Train NaNs=0, Test NaNs=0
Duplicates: handled by MedMNIST v2 official splits (no overlap between train/val/test)


## Model Factory Function

In [6]:
# MODEL FACTORY

def get_model(model_name, n_classes, input_shape=(224, 224, 3)):
    """Builds a model with the specified architecture."""
    inputs = layers.Input(shape=input_shape)

    if model_name == 'EfficientNetV2M':
        base = applications.EfficientNetV2M(include_top=False, weights='imagenet', input_tensor=inputs)
        x = layers.GlobalAveragePooling2D()(base.output)
        outputs = layers.Dense(n_classes, activation='softmax', dtype='float32')(x)

    elif model_name == 'InceptionResNetV2':
        x = layers.Rescaling(1./127.5, offset=-1)(inputs)
        base = applications.InceptionResNetV2(include_top=False, weights='imagenet', input_tensor=x)
        x = layers.GlobalAveragePooling2D()(base.output)
        outputs = layers.Dense(n_classes, activation='softmax', dtype='float32')(x)

    elif model_name == 'ConvNeXtBase':
        base = applications.ConvNeXtBase(include_top=False, weights='imagenet', input_tensor=inputs)
        x = layers.GlobalAveragePooling2D()(base.output)
        outputs = layers.Dense(n_classes, activation='softmax', dtype='float32')(x)

    elif model_name == 'ViT-Base':
        x = layers.Rescaling(1./127.5, offset=-1)(inputs)
        x = layers.Permute((3, 1, 2))(x)
        backbone = TFViTForImageClassification.from_pretrained(
            'google/vit-base-patch16-224', num_labels=n_classes,
            ignore_mismatched_sizes=True, use_safetensors=False)
        outputs = backbone.vit(x)[0]
        outputs = backbone.classifier(outputs[:, 0, :])
        outputs = layers.Activation('softmax', dtype='float32')(outputs)

    else:
        raise ValueError(f"Unknown Model: {model_name}")

    return models.Model(inputs=inputs, outputs=outputs, name=model_name)

## Trainer Function

In [7]:
# TRAINER

def train_and_save(model_name):
    print(f"\n{'='*20} TRAINING {model_name} {'='*20}")
    tf.keras.backend.clear_session()
    gc.collect()

    model_path = f"{OUTPUT_DIR}/{model_name}_{DATASET}.keras"
    val_pred_path = f"{OUTPUT_DIR}/{model_name}_val_preds.npy"
    test_pred_path = f"{OUTPUT_DIR}/{model_name}_test_preds.npy"

    if os.path.exists(model_path):
        print(f"Loading existing model: {model_path}")
        model = get_model(model_name, n_classes)
        model.load_weights(model_path)
        model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])
    else:
        model = get_model(model_name, n_classes)
        lr = LR_VIT if model_name == 'ViT-Base' else LR_CNN
        model.compile(optimizer=tf.keras.optimizers.Adam(lr),
                      loss='categorical_crossentropy', metrics=['accuracy'])

        cb = [
            ModelCheckpoint(model_path, save_best_only=True, monitor='val_accuracy', mode='max', verbose=1),
            ReduceLROnPlateau(monitor='val_loss', factor=REDUCE_FACTOR, patience=PATIENCE_REDUCE, min_lr=MIN_LR, verbose=1),
            EarlyStopping(monitor='val_accuracy', patience=PATIENCE_EARLY, restore_best_weights=True, verbose=1)
        ]
        model.fit(train_ds, validation_data=val_ds, epochs=EPOCHS, callbacks=cb, verbose=1)
        model.load_weights(model_path)

    # Save predictions
    print(f"Generating predictions...")
    val_preds = model.predict(val_ds, verbose=0)
    test_preds = model.predict(test_ds, verbose=0)
    np.save(val_pred_path, val_preds)
    np.save(test_pred_path, test_preds)

    acc = accuracy_score(y_true_flat, np.argmax(test_preds, axis=1))
    print(f"{model_name} Test Accuracy: {acc:.4f}")

    del model
    gc.collect()
    return acc

## Model Training

In [8]:
train_and_save('InceptionResNetV2')


==================== TRAINING InceptionResNetV2 ====================
219055592/219055592 [==============================] - 1s 0us/step
Epoch 1/30


I0000 00:00:1781239665.423936     194 cuda_dnn.cc:529] Loaded cuDNN version 91002


   1/4321 [..............................] - ETA: 80:43:10 - loss: 2.4845 - accuracy: 0.0000e+00

I0000 00:00:1781239668.027508     195 service.cc:152] XLA service 0x7d354c0963c0 initialized for platform CUDA (this does not guarantee that XLA will be used). Devices:
I0000 00:00:1781239668.027550     195 service.cc:160]   StreamExecutor device (0): Tesla T4, Compute Capability 7.5
I0000 00:00:1781239668.027554     195 service.cc:160]   StreamExecutor device (1): Tesla T4, Compute Capability 7.5
I0000 00:00:1781239668.511737     196 device_compiler.h:188] Compiled cluster using XLA!  This line is logged at most once for the lifetime of the process.


4321/4321 [==============================] - ETA: 0s - loss: 0.1472 - accuracy: 0.9522
Epoch 1: val_accuracy improved from -inf to 0.91665, saving model to /kaggle/working/organamnist_outputs/InceptionResNetV2_organamnist.keras
4321/4321 [==============================] - 630s 130ms/step - loss: 0.1472 - accuracy: 0.9522 - val_loss: 0.4396 - val_accuracy: 0.9167 - lr: 1.0000e-04
Epoch 2/30
4321/4321 [==============================] - ETA: 0s - loss: 0.0449 - accuracy: 0.9853
Epoch 2: val_accuracy did not improve from 0.91665
4321/4321 [==============================] - 486s 112ms/step - loss: 0.0449 - accuracy: 0.9853 - val_loss: 0.4642 - val_accuracy: 0.9011 - lr: 1.0000e-04
Epoch 3/30
4321/4321 [==============================] - ETA: 0s - loss: 0.0276 - accuracy: 0.9907
Epoch 3: val_accuracy improved from 0.91665 to 0.98521, saving model to /kaggle/working/organamnist_outputs/InceptionResNetV2_organamnist.keras
4321/4321 [==============================] - 495s 115ms/step - loss: 0.02

0.9721565980425245